# Notebook 05 — Module 2: PubMedBERT training across seeds

**ClinicalShield v2 · Revision pipeline · GPU required**

## Purpose
Train the detection classifier five times with different random seeds, and report every
metric as a mean with a standard deviation and confidence interval.

## Why five runs

Reviewer 3 raised this twice, in the review and again in the detailed report:

> *"The evaluation uses only one training run, so it may not fully show that the results are
> reliable and repeatable. The authors should repeat the experiments with several different
> random seeds and report statistics such as the mean, standard deviation, and confidence
> intervals."*

v1's Section 3.10 conceded the point in writing — *"All results reported in this paper are
from a single training run; variability across random seeds was not characterised"* — which
tells a reviewer the authors knew and did it anyway. Five seeds closes it properly.

## What else this notebook settles

**R3-Q8 — the checkpointing inconsistency.** v1 stopped early on validation *loss* but saved
the checkpoint with the best validation *accuracy*. Two criteria disagreeing about which
model is best. One criterion, validation loss, now governs both.

**R3-Q10 — attention localisation.** v1 showed a single attention heatmap, and its caption
contradicted the figure's own title. Here it is measured: precision@k and IoU of
high-attention tokens against the ground-truth injected spans NB02 recorded, with a random
baseline for comparison and failure cases printed.

**R4-3 — arithmetic consistency.** Accuracy, FPR and recall are recomputed from the pooled
confusion matrix and asserted equal to the reported values. v1 reported figures that did not
follow from its own matrix; here that cannot survive a run.

**Does Module 2 catch what Module 1 misses?** NB04 found four PyRIT variants passing through
ingestion — reversal, character transposition, random capitals, binary. Three leave the
payload as readable English, so the classifier is the layer that should catch them. Cell 8
reports recall per encoding and answers it.

## Runtime
GPU. Roughly 40–90 minutes for five seeds, depending on the card.

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    # Five independent training runs. Reviewer 3 asked for this twice: mean,
    # standard deviation and confidence intervals rather than a single number.
    "SEEDS": [13, 42, 101, 2024, 31337],

    "MODEL": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "MAX_LEN": 512,
    "BATCH_SIZE": 16,
    "LR": 2e-5,
    "WEIGHT_DECAY": 0.01,
    "EPOCHS": 5,
    "PATIENCE": 3,
    "CLASS_WEIGHTS": {0: 1.0, 1: 3.0},
    "GRAD_CLIP": 1.0,

    # Module 1 decodes before Module 2 classifies. Set False in NB11 to measure
    # what the decoding stage actually contributes.
    "APPLY_MODULE1": True,

    "SAVE_CHECKPOINTS": True,
    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
print("seeds  :", CONFIG["SEEDS"])
print("model  :", CONFIG["MODEL"])
print("epochs : %d, patience %d, batch %d"
      % (CONFIG["EPOCHS"], CONFIG["PATIENCE"], CONFIG["BATCH_SIZE"]))


In [ ]:
# ── Cell 2 · Environment, GPU, and NB03/NB04 inputs ───────────────────────────
import sys, os, json, random, hashlib, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "results": ROOT/"data"/"results", "src": ROOT/"src"/"clinicalshield",
        "ckpt": ROOT/"checkpoints", "figs": ROOT/"figures"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

for pkg, mod in [("transformers","transformers"), ("torch","torch"),
                 ("scikit-learn","sklearn"), ("unidecode","unidecode")]:
    try:
        __import__(mod)
    except ImportError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)

import torch
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("")
print("torch  :", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("gpu    :", torch.cuda.get_device_name(0),
          "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print("WARNING: no GPU. Five seeds on CPU will take many hours.")

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def restore(rel_path, label):
    local = ROOT / rel_path
    if local.exists():
        return local, "repo"
    if DRIVE_ROOT:
        src = DRIVE_ROOT / rel_path
        if src.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, local)
            return local, "drive"
    dm = str(DRIVE_ROOT / rel_path) if DRIVE_ROOT else "(drive not mounted)"
    raise FileNotFoundError("\n" + label + " not found.\n  repo : " + str(local)
                            + "\n  drive: " + dm + "\nRun NB03 first.")

p_ds, src_ds = restore("data/dataset/attack_dataset_split.jsonl", "attack_dataset_split.jsonl")
p_sp, _      = restore("data/stats/split_stats.json", "split_stats.json")

dataset     = read_jsonl(p_ds)
split_stats = json.load(open(p_sp))

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train","val","test"]}
for s in ["train","val","test"]:
    exp = split_stats["partitions"][s]["chunks"]
    assert len(parts[s]) == exp, "%s size %d != NB03 %d" % (s, len(parts[s]), exp)

print("")
print("partition    chunks   adversarial")
for s in ["train","val","test"]:
    a = sum(d["label"] for d in parts[s])
    print("  %-8s %6d   %5d (%.1f%%)" % (s, len(parts[s]), a, 100*a/len(parts[s])))
print("NB03 inputs verified")


## Ingestion before classification

Module 2 classifies the text Module 1 produces, which is the pipeline as specified. The
implementation is imported from `src/clinicalshield/module1.py` — the file NB04 exported —
rather than redefined here, so there is one implementation shared by the notebook, the
repository and the paper.

`APPLY_MODULE1` is switchable because NB11 turns it off to measure what the decoding stage
actually contributes. Reviewer 3's report asked for exactly that comparison.

In [ ]:
# ── Cell 3 · Module 1 ingestion, applied before classification ────────────────
# Module 2 sees decoded text, which is the pipeline as specified in Section 3.
# Importing from src/ rather than redefining keeps one implementation: the file
# NB04 exported is the file used here and the file in the repository.

m1_path = DIRS["src"] / "module1.py"
if not m1_path.exists():
    p_m1, _ = restore("src/clinicalshield/module1.py", "module1.py")
    m1_path = p_m1

import importlib.util
spec = importlib.util.spec_from_file_location("cs_module1", m1_path)
cs_module1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cs_module1)
module1 = cs_module1.module1
print("Module 1 loaded from", m1_path.relative_to(ROOT) if ROOT in m1_path.parents else m1_path)

if CONFIG["APPLY_MODULE1"]:
    t0 = time.time()
    for d in dataset:
        d["model_input"] = module1(d["text"])["decoded"]
    print("Module 1 applied to %d chunks in %.1fs" % (len(dataset), time.time()-t0))
    changed = sum(1 for d in dataset if d["model_input"] != d["text"])
    print("  chunks altered by ingestion: %d (%.1f%%)"
          % (changed, 100*changed/len(dataset)))
else:
    for d in dataset:
        d["model_input"] = d["text"]
    print("Module 1 BYPASSED (ablation mode)")

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train","val","test"]}


## Tokenisation

The domain-vocabulary argument is checked rather than asserted. If PubMedBERT genuinely
represents pharmacological terms as coherent units, `acetylcholinesterase` should tokenise
into few pieces — and the printed counts let a reader verify the claim instead of taking
it on trust.

In [ ]:
# ── Cell 4 · Dataset and model construction ───────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL"])
print("tokenizer:", CONFIG["MODEL"])

# The domain-vocabulary argument, checked rather than asserted
for term in ["acetylcholinesterase", "oligodeoxynucleotide", "contraindicated"]:
    print("  %-24s -> %d tokens" % (term, len(tokenizer.tokenize(term))))


class ChunkDataset(Dataset):
    def __init__(self, rows, tok, max_len):
        self.rows, self.tok, self.max_len = rows, tok, max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r["model_input"], truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(r["label"], dtype=torch.long)}


def make_loader(rows, shuffle, seed=None):
    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)
    return DataLoader(ChunkDataset(rows, tokenizer, CONFIG["MAX_LEN"]),
                      batch_size=CONFIG["BATCH_SIZE"], shuffle=shuffle,
                      generator=g if shuffle else None, num_workers=2,
                      pin_memory=(DEVICE == "cuda"))

print("")
print("loaders: train %d | val %d | test %d"
      % (len(parts["train"]), len(parts["val"]), len(parts["test"])))


## Training

Class weights penalise a missed attack three times more heavily than a false alarm, since in
a clinical setting the two errors are not equivalent.

Checkpoint selection and early stopping both use validation loss. That is the fix for
Reviewer 3's Q8: in v1 they used different criteria, so the saved model was not the model
the stopping rule had judged best.

In [ ]:
# ── Cell 5 · Training loop ────────────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix)

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    probs, preds, ys = [], [], []
    for b in loader:
        ids = b["input_ids"].to(DEVICE)
        am  = b["attention_mask"].to(DEVICE)
        out = model(input_ids=ids, attention_mask=am).logits
        p = torch.softmax(out, dim=-1)[:, 1]
        probs.extend(p.cpu().numpy())
        preds.extend(out.argmax(-1).cpu().numpy())
        ys.extend(b["labels"].numpy())
    ys, preds, probs = np.array(ys), np.array(preds), np.array(probs)
    pr, rc, f1, _ = precision_recall_fscore_support(ys, preds, average="binary",
                                                    zero_division=0)
    tn, fp, fn, tp = confusion_matrix(ys, preds, labels=[0, 1]).ravel()
    return {"accuracy": float(accuracy_score(ys, preds)),
            "precision": float(pr), "recall": float(rc), "f1": float(f1),
            "auc": float(roc_auc_score(ys, probs)) if len(set(ys)) > 1 else float("nan"),
            "fpr": float(fp / (fp + tn)) if (fp + tn) else 0.0,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "n": int(len(ys)),
            "probs": probs.tolist(), "preds": preds.tolist(), "labels": ys.tolist()}


def train_one_seed(seed, verbose=True):
    set_all_seeds(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["MODEL"], num_labels=2).to(DEVICE)

    tr = make_loader(parts["train"], shuffle=True, seed=seed)
    va = make_loader(parts["val"],   shuffle=False)

    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"],
                            weight_decay=CONFIG["WEIGHT_DECAY"])
    w = torch.tensor([CONFIG["CLASS_WEIGHTS"][0], CONFIG["CLASS_WEIGHTS"][1]],
                     dtype=torch.float).to(DEVICE)
    lossf = torch.nn.CrossEntropyLoss(weight=w)

    best_loss, best_state, bad, curves = float("inf"), None, 0, []
    for ep in range(1, CONFIG["EPOCHS"] + 1):
        model.train()
        run = 0.0
        for b in tr:
            opt.zero_grad()
            out = model(input_ids=b["input_ids"].to(DEVICE),
                        attention_mask=b["attention_mask"].to(DEVICE)).logits
            loss = lossf(out, b["labels"].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP"])
            opt.step()
            run += loss.item()
        tr_loss = run / len(tr)

        model.eval()
        vrun = 0.0
        with torch.no_grad():
            for b in va:
                out = model(input_ids=b["input_ids"].to(DEVICE),
                            attention_mask=b["attention_mask"].to(DEVICE)).logits
                vrun += lossf(out, b["labels"].to(DEVICE)).item()
        va_loss = vrun / len(va)
        vm = evaluate_model(model, va)
        curves.append({"epoch": ep, "train_loss": tr_loss, "val_loss": va_loss,
                       "val_acc": vm["accuracy"], "val_f1": vm["f1"]})
        if verbose:
            print("    epoch %d  train %.4f  val %.4f  acc %.4f  f1 %.4f"
                  % (ep, tr_loss, va_loss, vm["accuracy"], vm["f1"]))

        # Checkpoint and early stopping both on validation LOSS.
        # v1 stopped on loss but checkpointed on accuracy, which Reviewer 3
        # queried (Q8) as internally inconsistent. One criterion now governs both.
        if va_loss < best_loss - 1e-5:
            best_loss, bad = va_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CONFIG["PATIENCE"]:
                if verbose:
                    print("    early stop at epoch %d" % ep)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, curves, best_loss


## The five runs

The long cell. Each seed gets its own initialisation and data ordering, and all five are
evaluated on the same held-out test partition.

Checkpoints are saved per seed — Reviewer 4 found them missing from the v1 repository.

In [ ]:
# ── Cell 6 · Run all seeds ────────────────────────────────────────────────────
# The long cell. Five independent runs, each with its own initialisation and
# data order, evaluated on the same held-out test partition.

seed_runs = {}
t_start = time.time()

for seed in CONFIG["SEEDS"]:
    print("")
    print("=" * 62)
    print("SEED %d" % seed)
    print("=" * 62)
    t0 = time.time()
    model, curves, best_val = train_one_seed(seed)

    test_loader = make_loader(parts["test"], shuffle=False)
    tm = evaluate_model(model, test_loader)
    mins = (time.time() - t0) / 60

    print("  test  acc %.4f  prec %.4f  rec %.4f  f1 %.4f  auc %.4f  fpr %.4f"
          % (tm["accuracy"], tm["precision"], tm["recall"], tm["f1"],
             tm["auc"], tm["fpr"]))
    print("  confusion  tn %d  fp %d  fn %d  tp %d"
          % (tm["tn"], tm["fp"], tm["fn"], tm["tp"]))
    print("  %.1f min" % mins)

    seed_runs[seed] = {"curves": curves, "best_val_loss": best_val,
                       "test": tm, "minutes": mins,
                       "epochs_run": len(curves)}

    if CONFIG["SAVE_CHECKPOINTS"]:
        ck = DIRS["ckpt"] / ("module2_seed%d" % seed)
        model.save_pretrained(ck); tokenizer.save_pretrained(ck)
        print("  saved", ck.name)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("")
print("all seeds complete in %.1f min" % ((time.time() - t_start) / 60))


## Aggregation and the arithmetic gate

Every metric becomes mean, standard deviation and a t-based 95% confidence interval.

The consistency check at the end recomputes accuracy, FPR and recall directly from the
pooled confusion matrix and asserts they match the reported values. Reviewer 1 and Reviewer
4 independently caught v1 reporting 92.23% accuracy and 1.01% FPR against a confusion matrix
implying 86.98% and 10.83%. That mismatch cannot survive this cell.

In [ ]:
# ── Cell 7 · Aggregate across seeds ───────────────────────────────────────────
# Reviewer 3, twice: "repeat the experiments with several different random seeds
# and report statistics such as the mean, standard deviation, and confidence
# intervals." Every headline number in the paper becomes mean +/- SD with a CI.

from scipy import stats as sps

METRICS = ["accuracy", "precision", "recall", "f1", "auc", "fpr"]

def agg(values):
    v = np.array(values, dtype=float)
    n = len(v)
    mean, sd = float(v.mean()), float(v.std(ddof=1)) if n > 1 else 0.0
    if n > 1 and sd > 0:
        tcrit = sps.t.ppf(0.975, n - 1)
        half = tcrit * sd / math.sqrt(n)
    else:
        half = 0.0
    return {"mean": mean, "sd": sd, "ci_low": mean - half, "ci_high": mean + half,
            "min": float(v.min()), "max": float(v.max()), "n_seeds": int(n),
            "values": [float(x) for x in v]}

summary = {m: agg([seed_runs[s]["test"][m] for s in CONFIG["SEEDS"]])
           for m in METRICS}

print("HELD-OUT TEST, %d seeds" % len(CONFIG["SEEDS"]))
print("")
print("metric        mean      SD       95%% CI            range")
for m in METRICS:
    a = summary[m]
    print("  %-9s %7.4f  %7.4f  [%6.4f, %6.4f]  %6.4f-%6.4f"
          % (m, a["mean"], a["sd"], a["ci_low"], a["ci_high"], a["min"], a["max"]))

# pooled confusion matrix, summed across seeds
pooled = {k: int(sum(seed_runs[s]["test"][k] for s in CONFIG["SEEDS"]))
          for k in ["tn", "fp", "fn", "tp"]}
tot = sum(pooled.values())
print("")
print("pooled confusion across %d seeds (n=%d predictions)"
      % (len(CONFIG["SEEDS"]), tot))
print("  tn %d   fp %d   fn %d   tp %d" % (pooled["tn"], pooled["fp"],
                                            pooled["fn"], pooled["tp"]))

# Independent arithmetic check.
#
# Every seed is evaluated on the same test partition, so the benign count
# (fp + tn) and adversarial count (tp + fn) are identical across seeds. That
# makes the mean of per-seed rates and the rate computed from pooled counts
# mathematically equal, not merely close. The tolerance is therefore tight:
# any real discrepancy is a bug, not rounding.
#
# v1 reported 92.23% accuracy and 1.01% FPR against a confusion matrix implying
# 86.98% and 10.83%. Reviewer 1 and Reviewer 4 both found it. This cell makes
# that class of error impossible to ship.

TOL = 1e-4

acc_from_cm = (pooled["tn"] + pooled["tp"]) / tot
fpr_from_cm = pooled["fp"] / (pooled["fp"] + pooled["tn"])
rec_from_cm = pooled["tp"] / (pooled["tp"] + pooled["fn"])

print("")
print("CONSISTENCY CHECK — metrics recomputed from the confusion matrix")
print("  metric     reported    from matrix    delta        verdict")
checks = [("accuracy", summary["accuracy"]["mean"], acc_from_cm),
          ("fpr",      summary["fpr"]["mean"],      fpr_from_cm),
          ("recall",   summary["recall"]["mean"],   rec_from_cm)]
all_ok = True
for name, rep, cm in checks:
    delta = abs(rep - cm)
    ok = delta < TOL
    all_ok &= ok
    print("  %-9s  %.6f    %.6f     %.2e   %s"
          % (name, rep, cm, delta, "OK" if ok else "MISMATCH"))

for name, rep, cm in checks:
    assert abs(rep - cm) < TOL, (
        "%s inconsistent: reported %.6f, confusion matrix gives %.6f. "
        "This is the v1 failure mode; do not proceed." % (name, rep, cm))
print("  gate passed (tolerance %.0e)" % TOL)


## Breakdown by attack type

Where the layered-defence claim gets tested. If Module 2's recall on the variants Module 1
missed is high, the layers genuinely complement each other. If it is low, that is a real
limitation and belongs in the paper as one.

In [ ]:
# ── Cell 8 · Per-vector, per-category, per-encoding breakdown ─────────────────
# Does Module 2 catch what Module 1 misses? NB04 found four PyRIT variants
# passing through ingestion. Three leave the payload as readable English, so the
# classifier is the layer that should catch them. This is where that is tested.

test_rows = parts["test"]
by_seed_preds = {s: np.array(seed_runs[s]["test"]["preds"]) for s in CONFIG["SEEDS"]}
truth = np.array([r["label"] for r in test_rows])

def recall_over(mask, name):
    if mask.sum() == 0:
        return None
    vals = []
    for s in CONFIG["SEEDS"]:
        p = by_seed_preds[s][mask]
        t = truth[mask]
        vals.append(float((p[t == 1] == 1).mean()) if (t == 1).any() else float("nan"))
    a = agg(vals)
    return {"n": int(mask.sum()), **a}

breakdown = {}
for field in ["vector", "category", "encoding"]:
    breakdown[field] = {}
    vals = sorted({str(r[field]) for r in test_rows if r["label"] == 1})
    print("")
    print("recall by %s" % field)
    print("  %-26s n      mean      SD       95%% CI" % "value")
    for v in vals:
        mask = np.array([r["label"] == 1 and str(r[field]) == v for r in test_rows])
        res = recall_over(mask, v)
        if res is None:
            continue
        breakdown[field][v] = res
        print("  %-26s %4d   %7.4f  %7.4f  [%6.4f, %6.4f]"
              % (v, res["n"], res["mean"], res["sd"], res["ci_low"], res["ci_high"]))


## Attention localisation

Reviewer 3, Q10, asked for quantitative localisation metrics, a comparison against
alternatives, and failure cases. This cell delivers the first and third.

**precision@k** — of the k highest-attention tokens, how many fall inside the true injected
span, with k set to the payload's own token length so a perfect localiser scores 1.0.
**IoU** — overlap between the attended set and the true span. **Random baseline** — what
chance alone would score on the same texts, which is the number that makes the result
interpretable.

Failure cases are printed rather than hidden. A comparison against Integrated Gradients
follows in NB11; this establishes the measurement first.

In [ ]:
# ── Cell 9 · Attention localisation (R3-Q10) ──────────────────────────────────
# Reviewer 3, Q10: "add quantitative tests showing how well the model finds
# malicious tokens ... report localization metrics, compare with other
# interpretability methods, and include failure cases."
#
# NB02 recorded the exact character span of every injected payload, so this can
# be measured rather than illustrated with one example as v1 did.

ck = DIRS["ckpt"] / ("module2_seed%d" % CONFIG["SEEDS"][0])
loc_model = AutoModelForSequenceClassification.from_pretrained(
    ck, num_labels=2, output_attentions=True).to(DEVICE).eval()

def token_spans(text):
    enc = tokenizer(text, truncation=True, max_length=CONFIG["MAX_LEN"],
                    return_offsets_mapping=True, return_tensors="pt")
    return enc, enc["offset_mapping"][0].tolist()

@torch.no_grad()
def attention_scores(text):
    enc, offsets = token_spans(text)
    ids = enc["input_ids"].to(DEVICE)
    am  = enc["attention_mask"].to(DEVICE)
    out = loc_model(input_ids=ids, attention_mask=am)
    # final layer, averaged over heads, attention received from [CLS]
    att = out.attentions[-1][0].mean(0)[0]
    return att.cpu().numpy(), offsets

def localisation(rows, k_frac=1.0, limit=200):
    """Precision@k over tokens: of the k highest-attention tokens, how many fall
    inside the true injected span? k is set to the payload's own token length,
    so a perfect localiser scores 1.0."""
    ious, precs, misses = [], [], []
    sel = [r for r in rows if r["label"] == 1 and r.get("span_start") is not None][:limit]
    for r in sel:
        txt = r["text"]
        s0, e0 = r["span_start"], r["span_end"]
        att, offs = attention_scores(txt)
        cand = [(i, a) for i, a in enumerate(att)
                if offs[i][1] > offs[i][0]]
        if not cand:
            continue
        inside = [i for i, _ in cand if offs[i][0] >= s0 and offs[i][1] <= e0]
        if not inside:
            continue
        k = max(1, int(len(inside) * k_frac))
        top = [i for i, _ in sorted(cand, key=lambda x: -x[1])[:k]]
        hit = len(set(top) & set(inside))
        precs.append(hit / k)
        union = len(set(top) | set(inside))
        ious.append(hit / union if union else 0.0)
        if hit / k < 0.2:
            misses.append({"chunk_id": r["chunk_id"], "vector": r["vector"],
                           "encoding": r["encoding"],
                           "precision_at_k": hit / k,
                           "top_tokens": [txt[offs[i][0]:offs[i][1]] for i in top[:8]]})
    return {"n": len(precs),
            "precision_at_k_mean": float(np.mean(precs)) if precs else 0.0,
            "precision_at_k_sd": float(np.std(precs, ddof=1)) if len(precs) > 1 else 0.0,
            "iou_mean": float(np.mean(ious)) if ious else 0.0,
            "random_baseline": None, "failures": misses[:10]}

loc = localisation(test_rows)

# random baseline: what precision@k would chance give on the same texts?
rng_loc = random.Random(CONFIG["SEEDS"][0])
sel = [r for r in test_rows if r["label"] == 1 and r.get("span_start") is not None][:200]
rand_prec = []
for r in sel:
    _att, offs = attention_scores(r["text"])
    valid = [i for i in range(len(offs)) if offs[i][1] > offs[i][0]]
    inside = [i for i in valid if offs[i][0] >= r["span_start"] and offs[i][1] <= r["span_end"]]
    if not inside:
        continue
    k = len(inside)
    pick = rng_loc.sample(valid, min(k, len(valid)))
    rand_prec.append(len(set(pick) & set(inside)) / k)
loc["random_baseline"] = float(np.mean(rand_prec)) if rand_prec else 0.0

print("ATTENTION LOCALISATION — seed %d, n=%d adversarial chunks"
      % (CONFIG["SEEDS"][0], loc["n"]))
print("  precision@k   %.4f  (SD %.4f)" % (loc["precision_at_k_mean"],
                                            loc["precision_at_k_sd"]))
print("  IoU           %.4f" % loc["iou_mean"])
print("  random chance %.4f" % loc["random_baseline"])
lift = (loc["precision_at_k_mean"] / loc["random_baseline"]
        if loc["random_baseline"] else float("inf"))
print("  lift over chance  %.1fx" % lift)
print("")
print("  failure cases (precision@k < 0.2): %d" % len(loc["failures"]))
for f in loc["failures"][:3]:
    print("    %s  %s/%s  p@k %.2f  top: %s"
          % (f["chunk_id"], f["vector"], f["encoding"], f["precision_at_k"],
             " ".join(f["top_tokens"][:5])))

del loc_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()


## Persist

`module2_results.json` holds the aggregates and goes into git. `module2_predictions.json`
holds raw per-seed predictions so NB12 can recompute every table from source rather than
transcribing numbers — which is how v1's tables drifted from its figures.

In [ ]:
# ── Cell 10 · Save results ────────────────────────────────────────────────────
def strip_arrays(d):
    """Per-prediction arrays are large; keep them out of the committed JSON."""
    out = dict(d)
    for k in ["probs", "preds", "labels"]:
        out.pop(k, None)
    return out

module2_results = {
    "notebook": "05_module2_training",
    "model": CONFIG["MODEL"],
    "seeds": CONFIG["SEEDS"],
    "config": {k: v for k, v in CONFIG.items()
               if k not in ["USE_DRIVE", "DRIVE_DIR", "PUSH_TO_GITHUB",
                            "GITHUB_REPO", "GITHUB_BRANCH",
                            "GIT_USER_NAME", "GIT_USER_EMAIL"]},
    "protocol": {
        "checkpoint_criterion": "validation loss",
        "early_stopping_criterion": "validation loss",
        "note": ("v1 stopped on validation loss but checkpointed on validation "
                 "accuracy. Reviewer 3 (Q8) queried that inconsistency; one "
                 "criterion now governs both."),
        "module1_applied": CONFIG["APPLY_MODULE1"],
    },
    "summary": summary,
    "pooled_confusion": pooled,
    "consistency_check": {
        "accuracy_from_confusion": float(acc_from_cm),
        "fpr_from_confusion": float(fpr_from_cm),
        "recall_from_confusion": float(rec_from_cm),
        "passed": True,
    },
    "per_seed": {str(s): {"test": strip_arrays(seed_runs[s]["test"]),
                          "curves": seed_runs[s]["curves"],
                          "epochs_run": seed_runs[s]["epochs_run"],
                          "best_val_loss": seed_runs[s]["best_val_loss"],
                          "minutes": seed_runs[s]["minutes"]}
                 for s in CONFIG["SEEDS"]},
    "breakdown": breakdown,
    "localisation": loc,
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}

with open(DIRS["results"] / "module2_results.json", "w") as f:
    json.dump(module2_results, f, indent=2)

# raw predictions kept separately for NB12 to recompute every table from source
preds_out = {str(s): {"preds": seed_runs[s]["test"]["preds"],
                      "probs": seed_runs[s]["test"]["probs"]}
             for s in CONFIG["SEEDS"]}
preds_out["chunk_ids"] = [r["chunk_id"] for r in parts["test"]]
preds_out["labels"] = [r["label"] for r in parts["test"]]
with open(DIRS["results"] / "module2_predictions.json", "w") as f:
    json.dump(preds_out, f)

if IN_COLAB and DRIVE_ROOT:
    for sub in ["results", "stats"]:
        dst = DRIVE_ROOT / "data" / sub; dst.mkdir(parents=True, exist_ok=True)
        for f_ in (ROOT / "data" / sub).glob("*"):
            shutil.copy2(f_, dst / f_.name)
    if CONFIG["SAVE_CHECKPOINTS"]:
        dst = DRIVE_ROOT / "checkpoints"; dst.mkdir(parents=True, exist_ok=True)
        for ckdir in DIRS["ckpt"].glob("module2_seed*"):
            tgt = dst / ckdir.name; tgt.mkdir(exist_ok=True)
            for f_ in ckdir.glob("*"):
                shutil.copy2(f_, tgt / f_.name)
        print("checkpoints mirrored to Drive")
    print("results mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    keep = ["data/results/module2_results.json"]
    subprocess.run(["git","-C",str(ROOT),"add","-f"] + keep, check=False)
    st = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        msg = ("NB05: Module 2 trained on %d seeds, test acc %.4f +/- %.4f"
               % (len(CONFIG["SEEDS"]), summary["accuracy"]["mean"],
                  summary["accuracy"]["sd"]))
        subprocess.run(["git","-C",str(ROOT),"commit","-q","-m",msg], check=False)
        pr = subprocess.run(["git","-C",str(ROOT),"push","-q","origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])

print("written: module2_results.json, module2_predictions.json")


## Summary

**Paste the output below into the chat.**

What I will check first: the arithmetic consistency line, the standard deviations (large
spread across seeds would mean the model is unstable), and recall on the encodings Module 1
let through.

In [ ]:
# ── Cell 11 · NB05 SUMMARY — paste this output ────────────────────────────────
print("=" * 68)
print("NB05 — MODULE 2: PubMedBERT TRAINING ACROSS SEEDS")
print("=" * 68)
print("model   : %s" % CONFIG["MODEL"].split("/")[-1])
print("seeds   : %s" % CONFIG["SEEDS"])
print("device  : %s" % DEVICE)
print("Module 1 applied before classification : %s" % CONFIG["APPLY_MODULE1"])
print("checkpoint and early stopping both on  : validation loss  (R3-Q8)")
print("")
print("HELD-OUT TEST, mean +/- SD across %d seeds" % len(CONFIG["SEEDS"]))
print("metric        mean      SD       95% CI            range")
for m in METRICS:
    a = summary[m]
    print("  %-9s %7.4f  %7.4f  [%6.4f, %6.4f]  %6.4f-%6.4f"
          % (m, a["mean"], a["sd"], a["ci_low"], a["ci_high"], a["min"], a["max"]))
print("")
print("pooled confusion (n=%d)" % sum(pooled.values()))
print("  tn %d   fp %d   fn %d   tp %d"
      % (pooled["tn"], pooled["fp"], pooled["fn"], pooled["tp"]))
print("  accuracy from matrix %.4f | fpr %.4f | recall %.4f  -> consistent"
      % (acc_from_cm, fpr_from_cm, rec_from_cm))
print("")
print("RECALL BY ATTACK VECTOR")
for v, r in sorted(breakdown["vector"].items()):
    print("  %-18s n=%4d  %.4f +/- %.4f" % (v, r["n"], r["mean"], r["sd"]))
print("")
print("RECALL BY ENCODING  (does Module 2 catch what Module 1 missed?)")
for v, r in sorted(breakdown["encoding"].items(), key=lambda x: x[1]["mean"]):
    print("  %-18s n=%4d  %.4f +/- %.4f" % (v, r["n"], r["mean"], r["sd"]))
print("")
print("ATTENTION LOCALISATION (R3-Q10)")
print("  precision@k     %.4f +/- %.4f  (n=%d)"
      % (loc["precision_at_k_mean"], loc["precision_at_k_sd"], loc["n"]))
print("  IoU             %.4f" % loc["iou_mean"])
print("  random baseline %.4f" % loc["random_baseline"])
print("  failure cases   %d" % len(loc["failures"]))
print("")
print("TRAINING")
for s in CONFIG["SEEDS"]:
    r = seed_runs[s]
    print("  seed %-6d epochs %d  best val loss %.4f  %.1f min"
          % (s, r["epochs_run"], r["best_val_loss"], r["minutes"]))
print("=" * 68)
print("NB05 COMPLETE — ready for NB06 (Module 3 intent verification)")
print("=" * 68)
